# 【notebook｜N-18】Python 语言 I

> 对应理论篇：[`PY1-Python语言I.md`](../../00-预备/PY1-Python语言I.md)（doccode `PY1`）
> 对应代码：[`py1_language_basics.py`](../../00-预备/code/py1_language_basics.py)（Tag：`F-pre-py1`）
> 覆盖条目：【定义 PY1.1.1｜D-PY1.1.1】执行模型、【定义 PY1.2.1｜D-PY1.2.1】名字绑定、【定义 PY1.3.6｜D-PY1.3.6】dict/set、【注 PY1.3.7｜R-PY1.3.7】切片、【定义 PY1.4.1｜D-PY1.4.1】真值测试、【定义 PY1.5.5｜D-PY1.5.5】闭包、【定义 PY1.6.3｜D-PY1.6.3】迭代协议。

本 notebook 面向有 C/C++ 经验、Python 零基础的读者；每节先看理论篇对应条目，再运行下面的 cell 核对行为。


## 1. 执行模型与三种入口

In [ ]:
# 【N-18.c1】
import sys, dis
print("Python:", sys.version.split()[0])
print("本文件运行时 __name__ =", __name__, "（脚本入口为 '__main__'，被导入时为模块名）")

def add(a, b):
    return a + b

print("add 的字节码指令:", [i.opname for i in dis.get_instructions(add)])

## 2. 名字是绑定，不是盒子（对照 C 变量/指针）

In [ ]:
# 【N-18.c2】
a = [1, 2, 3]
b = a                 # b 与 a 同一对象
b.append(4)
print("a =", a, "| a is b:", a is b, "| 身份相同会一起变")

c = a[:]              # 切片是浅拷贝（新对象）
c.append(5)
print("a =", a, "| c =", c, "| a is c:", a is c)

s = "abc"
t = s
t = t + "d"           # 不可变对象：重绑定，不改 s
print("s =", s, "| t =", t)

## 3. 内置类型：数值、文本与字节

In [ ]:
# 【N-18.c3】
print("int 任意精度:", 2 ** 100)
print("真除法 7/2 =", 7 / 2, "| 向下取整 7//2 =", 7 // 2, "| 取模 -7%3 =", -7 % 3)
print("bool 是 int 子类:", isinstance(True, int), True == 1, False == 0)
print("None 单例:", None is None)

text = "汉字"
raw = text.encode("utf-8")
print("str 长度", len(text), "| utf-8 字节数", len(raw), "| 解码回:", raw.decode("utf-8"))

## 4. dict / set 与哈希约束

In [ ]:
# 【N-18.c4】
d = {"a": 1, "b": 2}          # 3.7+ 保持插入序
d.setdefault("a", 99)          # 已存在则不覆盖
d.setdefault("c", 3)
print("dict:", d, "| keys 保序:", list(d))

try:
    {[1], [2]}                 # list 不可哈希，不能进 set
except TypeError as e:
    print("set 拒绝 list:", str(e)[:60])

s = {1, 2, 3}
print("set 成员测试 2 in s:", 2 in s, "| frozenset 可作键:", {frozenset({1, 2}): "x"})

## 5. 切片：步长、负索引、越界截断（对照 PY4/PY5 的视图语义）

In [ ]:
# 【N-18.c5】
xs = [0, 1, 2, 3, 4, 5]
print("xs[1:4]  =", xs[1:4])
print("xs[::-1] =", xs[::-1], "| xs[::2] =", xs[::2], "| xs[-1] =", xs[-1])
print("越界 xs[10:20] =", xs[10:20], "（不报错，按边界截断）")
print("切片是拷贝，不是视图；numpy/torch 的切片才是视图，见 PY4/PY5")

## 6. 控制流：真值与 for-else、海象

In [ ]:
# 【N-18.c6】
print("真值测试:", bool([]), bool([0]), bool(0), bool('0'), bool(None))

target = 3
for x in [1, 2, 4]:
    if x == target:
        break
else:
    print("for-else: 未找到", target)

f = iter(["line1", "line2"])
while (line := next(f, None)):
    print("海象读到:", line)

## 7. match 结构化模式匹配

In [ ]:
# 【N-18.c7】
def classify(value):
    match value:
        case 0:
            return "zero"
        case int() as n if n < 0:
            return "neg"
        case [_, _, _]:
            return "triple"
        case {"kind": k}:
            return f"mapping:{k}"
        case _:
            return "other"

print([classify(v) for v in (0, -3, [1, 2, 3], {"kind": "k"}, "s")])

## 8. 函数：默认值陷阱与参数形态

In [ ]:
# 【N-18.c8】
def bad(x, acc=[]):        # 陷阱：acc 在 def 时创建一次，被所有调用共享
    acc.append(x)
    return acc

def good(x, acc=None):     # 修正：None 哨兵 + 函数体内新建
    if acc is None:
        acc = []
    acc.append(x)
    return acc

print("bad 三次:", bad(1), bad(2), bad(3))
print("good 三次:", good(1), good(2), good(3))

def tag(name, *items, sep="-", **meta):
    body = sep.join([name, *items])
    return body + (":" + ",".join(f"{k}={v}" for k, v in meta.items()) if meta else "")

print(tag("a", "b", "c", sep="/"), "|", tag("a", x=1, y=2))

## 9. 闭包与 nonlocal

In [ ]:
# 【N-18.c9】
def make_counter(start=0):
    n = start
    def tick():
        nonlocal n     # 必须：内层要重新绑定外层变量
        n += 1
        return n
    return tick

c = make_counter()
print("counter:", c(), c(), c())

## 10. 推导式、生成器表达式与迭代协议

In [ ]:
# 【N-18.c10】
xs = [1, 2, 3, 4]
print("列表推导:", [x * x for x in xs if x % 2 == 0])
print("字典推导:", {x: x * x for x in xs})

gen = (x * x for x in xs)       # 惰性迭代器，一次性耗尽
print("next(gen) =", next(gen), "| list(gen) =", list(gen))

it = iter(xs)
print("iter/next:", next(it))
try:
    next(iter([]))
except StopIteration:
    print("空迭代器抛 StopIteration")

## 11. 自测（对照理论篇 §8）

1. `b = a; b += [3]` 与 `b = b + [3]` 对 `a` 的影响为何不同？（【练习 PY1.8.2｜Ex-PY1.8.2】）
2. `m = [[0] * 2] * 2; m[0][0] = 1` 后 `m` 是什么？为什么？（【练习 PY1.8.3｜Ex-PY1.8.3】）
3. 用 `for-else` 重写“查找未果则报告”。（【练习 PY1.8.5｜Ex-PY1.8.5】）
4. 写 `make_accumulator(start)` 并解释 `nonlocal` 的作用。（【练习 PY1.8.7｜Ex-PY1.8.7】）
